In [1]:
%load_ext autoreload
%autoreload 2
%load_ext line_profiler

import numpy as np
import matplotlib.pyplot as plt
import qutip as qt
from scipy.optimize import minimize, LinearConstraint, Bounds
import qutip_qtrl.pulseoptim as cpo
from types import MethodType
from time import time

from quspin.basis import spin_basis_1d
from quspin.operators import hamiltonian

import geodesiq as gq

# Landau-Zener

In [12]:
# ----- Define ControlModel -----
def hamiltonian_LZ(z, x):
    return np.array([[z, x], [x, -z]])

def hamiltonian_LZ_derivative(z, x):
    return np.array([[1, 0], [0, -1]])


n_rep = 5
n_ts = 2 ** 7 + 1  # time slices
amp_lbound, amp_ubound = -10.0, 10.0
x = 1
tf = 2

alpha = 2
beta = 2

fid_err_targ = 1e-8

index_0 = 0
index_t = 0

In [13]:
def solve_GRAPE(x, n_ts):
    H_initial = qt.Qobj(hamiltonian_LZ(amp_lbound, x))
    H_final = qt.Qobj(hamiltonian_LZ(amp_ubound, x))

    psi0 = H_initial.eigenstates()[1][index_0]
    psit = H_final.eigenstates()[1][index_t]

    H_c = qt.Qobj(hamiltonian_LZ(1, 0))
    H_d = qt.Qobj(hamiltonian_LZ(0, x))
    
    optim_GRAPE = cpo.create_pulse_optimizer(H_d, [H_c], psi0, psit, num_tslots=n_ts,
                                         evo_time=tf, amp_lbound=amp_lbound,
                                         amp_ubound=amp_ubound, fid_err_targ=fid_err_targ,
                                         max_iter=2000, alg="GRAPE", dyn_type="UNIT",)

    dyn = optim_GRAPE.dynamics
    dyn.init_timeslots()
    
    # Initial guess: linear ramp
    init_amps = np.linspace(amp_lbound, amp_ubound, n_ts)[:, None]
    # init_amps = np.random.rand(n_ts)[:, None]
    dyn.initialize_controls(init_amps)
    
    def fixed_endpoint_bounds(self):
        bounds = [(amp_lbound, amp_ubound)] * n_ts
        bounds[0] = (amp_lbound, amp_lbound)
        bounds[-1] = (amp_ubound, amp_ubound)
        return bounds
    
    optim_GRAPE._build_bounds_list = MethodType(fixed_endpoint_bounds, optim_GRAPE)
    
    result_GRAPE = optim_GRAPE.run_optimization()

In [14]:
def solve_gq(x, n_ts):
    H_initial = qt.Qobj(hamiltonian_LZ(amp_lbound, x))
    H_final = qt.Qobj(hamiltonian_LZ(amp_ubound, x))
    
    psi0 = H_initial.eigenstates()[1][index_0]
    psit = H_final.eigenstates()[1][index_t]
    
    H_c = qt.Qobj(hamiltonian_LZ(1, 0))
    H_d = qt.Qobj(hamiltonian_LZ(0, x))

    model = gq.ControlModel(hamiltonian_LZ, partial_H_func=hamiltonian_LZ_derivative)
    model.set_parameters(x=x)
    model.set_control(control_name='z', pulse_initial=amp_lbound, pulse_final=amp_ubound,
                      initial_state=index_0, alpha=alpha, beta=beta, num_steps=n_ts)

    model.solve_problem(pulse_accuracy=1000)

    dynamics = gq.Dynamics(tf, model)
    gq_error = 1 - np.sqrt(dynamics.state_fidelity())

In [15]:
t0 = time()

for _ in range(n_rep):
    solve_GRAPE(x, n_ts)

GRAPE_time = (time() - t0) / n_rep
print(f'{GRAPE_time=}')

GRAPE_time=0.1754138946533203


In [16]:
t0 = time()

for _ in range(n_rep):
    solve_gq(x, n_ts)

gq_time = (time() - t0) / n_rep
print(f'{gq_time=}')

gq_time=0.1651996612548828


# Many body

In [2]:
# ----- Define ControlModel -----
def ising_model(lam, L, hx, hz):
    """
    Constructs the Ising ControlModel with transverse (hx) and longitudinal (hz) fields.
    """
    zz_list = [[lam, i, i + 1] for i in range(L - 1)]
    z_list = [[lam * hz, i] for i in range(L)]
    x_list = [[(1 - lam) * hx, i] for i in range(L)]

    static = [["zz", zz_list], ["z", z_list], ["x", x_list]]

    basis = spin_basis_1d(L, pblock=1)
    H = hamiltonian(static, [], basis=basis, dtype=np.float64, check_symm=False, check_herm=False)

    return H.toarray()

def ising_model_derivative(lam, L, hx, hz):
    zz_list = [[1.0, i, i + 1] for i in range(L - 1)]
    z_list = [[hz, i] for i in range(L)]
    x_list = [[-hx, i] for i in range(L)]

    static = [["zz", zz_list], ["z", z_list], ["x", x_list]]

    basis = spin_basis_1d(L, pblock=1)
    dH = hamiltonian(static, [], basis=basis, dtype=np.float64, check_symm=False, check_herm=False)

    return dH.toarray()

n_rep = 5
n_ts = 2 ** 10 + 1  # time slices
amp_lbound, amp_ubound = 0.1, 0.9
L, hx, hz = 4, 1, .8
tf = 250

alpha = 2
beta = 2

fid_err_targ = 0.00031

index_0 = 6
index_t = 6

In [3]:
def solve_GRAPE(hx, n_ts):
    H_initial = qt.Qobj(ising_model(amp_lbound, L, hx, hz))
    H_final = qt.Qobj(ising_model(amp_ubound, L, hx, hz))

    psi0 = H_initial.eigenstates()[1][index_0]
    psit = H_final.eigenstates()[1][index_t]

    H_d = qt.Qobj(ising_model(0, L, hx, hz))
    H_c = qt.Qobj(ising_model(1, L, hx, hz)) - H_d
    
    optim_GRAPE = cpo.create_pulse_optimizer(H_d, [H_c], psi0, psit, num_tslots=n_ts,
                                             evo_time=tf, amp_lbound=amp_lbound,
                                             amp_ubound=amp_ubound, fid_err_targ=fid_err_targ,
                                             max_iter=2000, alg="GRAPE", dyn_type="UNIT",)

    dyn = optim_GRAPE.dynamics
    dyn.init_timeslots()
    
    # Initial guess: linear ramp
    init_amps = np.linspace(amp_lbound, amp_ubound, n_ts)[:, None]
    # init_amps = np.random.rand(n_ts)[:, None]
    dyn.initialize_controls(init_amps)
    
    def fixed_endpoint_bounds(self):
        bounds = [(amp_lbound, amp_ubound)] * n_ts
        bounds[0] = (amp_lbound, amp_lbound)
        bounds[-1] = (amp_ubound, amp_ubound)
        return bounds
    
    optim_GRAPE._build_bounds_list = MethodType(fixed_endpoint_bounds, optim_GRAPE)
    
    result_GRAPE = optim_GRAPE.run_optimization()

In [4]:
def solve_gq(hx, n_ts):
    H_initial = qt.Qobj(ising_model(amp_lbound, L, hx, hz))
    H_final = qt.Qobj(ising_model(amp_ubound, L, hx, hz))
    
    psi0 = H_initial.eigenstates()[1][index_0]
    psit = H_final.eigenstates()[1][index_t]

    model = gq.ControlModel(ising_model, partial_H_func=ising_model_derivative)
    model.set_parameters(L=L, hx=hx, hz=hz)
    model.set_control(control_name='lam', pulse_initial=amp_lbound, pulse_final=amp_ubound,
                      initial_state=index_0, alpha=alpha, beta=beta, num_steps=n_ts)

    model.solve_problem(pulse_accuracy=n_ts)

    dynamics = gq.Dynamics(tf, model)
    gq_error = 1 - np.sqrt(dynamics.state_fidelity())

In [55]:
t0 = time()

for _ in range(n_rep):
    solve_GRAPE(hx, n_ts)

GRAPE_time = (time() - t0) / n_rep
print(f'{GRAPE_time=}')

GRAPE_time=1.0805298805236816


In [7]:
%lprun -f solve_gq -f gq.ControlModel._solve_eigenproblem -f gq.ControlModel._call_hamiltonian -f gq.decompose_hamiltonian solve_gq(hx, n_ts)

Timer unit: 1e-09 s

Total time: 8.00777 s
File: /tmp/ipykernel_675010/3145559604.py
Function: solve_gq at line 1

Line #      Hits         Time  Per Hit   % Time  Line Contents

Total time: 5.01354 s
File: /mnt/d/OneDrive-UniA/Projects/geodesiq-control/src/geodesiq/controlmodel.py
Function: ControlModel._solve_eigenproblem at line 680

Line #      Hits         Time  Per Hit   % Time  Line Contents
   680                                               def _solve_eigenproblem(self, config: _EigensystemParameters | None = None) -> None:
   681                                                   """Solve the Hamiltonian eigenproblem over a validated control grid."""
   682         1      10770.0  10770.0      0.0          if self._flags["eigenproblem_solved"]:
   683                                                       return
   684                                           
   685         1        420.0    420.0      0.0          if config is None:
   686                                   

In [6]:
t0 = time()

for _ in range(n_rep):
    solve_gq(hx, n_ts)

gq_time = (time() - t0) / n_rep
print(f'{gq_time=}')

gq_time=2.929389762878418
